#### Tools

Models can request to call tools that perform tasks such as fetching data from a database,searching the web or running code.Tools are pairings of :
1. A schema including the name of the tool,a description,and/or argument definitions (often as Json Schema)
2. A function or coroutine to execute

In [1]:
import os
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-20b")
response = model.invoke("Write a short poem on cricket")
print(response.content)

In twilight’s hush the field is set—  
A willow’s whisper, leather met.  
A ball, a bat, a fleeting dance,  
Where thunder roars in every glance.  

Runs like rivers, wickets fall—  
The crowd breathes, the night stands tall.  
Cricket’s pulse in heartbeats cast,  
A timeless game of present, past.


In [2]:
#tools
 
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """ Get the weather at a location"""
    return f"it's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("whats the weather in bangalore?")
print(response)

content='' additional_kwargs={'reasoning_content': 'The user asks: "whats the weather in bangalore?" We should use the get_weather function.', 'tool_calls': [{'id': 'fc_4932d9b9-6903-44ce-922e-9457100f43bd', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 128, 'total_tokens': 174, 'completion_time': 0.047920207, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.006272423, 'prompt_tokens_details': None, 'queue_time': 0.27933794, 'total_time': 0.05419263}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_66891002f6', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a052f1-8793-7ee0-a152-34d186a892fd-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'fc_4932d9b9-6903-44ce-922e-9457100f43bd', 'type': 'tool_call'}] invalid_tool_call

In [4]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': 'fc_4932d9b9-6903-44ce-922e-9457100f43bd',
  'type': 'tool_call'}]

### Tool execution loop

In [5]:

#Step 1: Model generates the tool calls
messages = [{"role":"user","content":"what's the weather in bangalore?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


#Step 2:Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


#Step 3:Pass the results back to the model for final responses
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It’s sunny in Bangalore.


In [6]:
messages

[{'role': 'user', 'content': "what's the weather in bangalore?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the function get_weather.', 'tool_calls': [{'id': 'fc_7538814c-9274-4da5-a16c-a992cc10b3c8', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 128, 'total_tokens': 162, 'completion_time': 0.045212396, 'completion_tokens_details': {'reasoning_tokens': 10}, 'prompt_time': 0.006594989, 'prompt_tokens_details': None, 'queue_time': 0.166057021, 'total_time': 0.051807385}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a052f1-893a-74a2-b45d-75afeb967d59-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'fc_7538814c-9274-4da5-a16c-a992cc10b3c8', 'typ